In [1]:
# to import things from script folder on this notebooks
import sys; sys.path.append('..')
from Bio import AlignIO, SeqIO
import pandas as pd
from pathlib import Path
import itertools as it

from scripts.conf import DATA_RAW, DATA_INTERIM, DATA_PROCESSED

In [2]:
sars2protein_folder = DATA_PROCESSED/'sars2_proteins'

In [18]:
def filter_Xchar(proteins,th=5.0):
    """
    Filter seqs with non-protein character `X` percentage >= threshold
    """
    
    l = lambda item: item.seq.count('X')/len(item.seq)*100 <= th 
    return filter(l,proteins)

In [19]:
def split_records(records, n_max=20_000):
    """ 
    Split fasta records into equal number of n_max seqs.
    Returns an iterator.
    """
    max_n_mafft = 20_000
    n_start = 0
    for i,l in enumerate(it.tee(records,5)):
        l_slice = it.islice(l,n_start,max_n_mafft)
        n_start+=20_000
        max_n_mafft +=20_000
        
        yield l_slice

In [31]:
inputfile = sars2protein_folder/'sars2_others_Spike.fasta'


In [32]:

proteins = SeqIO.parse(inputfile,"fasta")

filter1 = filter_Xchar(proteins, th=5)


In [33]:
for i, records in enumerate(split_records(filter1, n_max=20_000),1):
    outfile = samplefile.parent/samplefile.name.replace(f'.fasta',f'_split{i}.fasta')
    
    with open(outfile,'w') as f:
        for rec in records:
            f.write(f'>{rec.description}\n')
            f.write(f'{rec.seq}\n')